# 02 · Silver · `inspecoes`

Lê `v_energy.bronze.inspecoes`, normaliza a planilha preenchida manualmente e grava a Silver. A Bronze já uniu as três abas e preservou `aba_origem`.

**Grão:** uma linha por inspeção.

As contagens são descobertas na execução. O notebook não assume que os nomes observados no Excel estejam exatamente padronizados.

## 1. Parâmetros do ambiente

In [ ]:
catalog = "v_energy"
gold_schema = f"{catalog}.gold"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
landing_path = f"/Volumes/{catalog}/bronze/landing"
tabela_bronze = f"{bronze_schema}.inspecoes"
tabela_silver = f"{silver_schema}.inspecoes"
from datetime import datetime
from pyspark.sql import functions as F, Row
bronze = spark.table(tabela_bronze)
bronze.printSchema()
print(f"Linhas na Bronze: {bronze.count():,}".replace(",", "."))
display(bronze.limit(10))

## 2. Schema real e aliases controlados

As colunas são resolvidas contra o schema real. Se uma coluna estrutural não existir, a execução para para evitar uma transformação silenciosamente incorreta.

In [ ]:
def coluna_existente(df, aliases, obrigatoria=True):
    encontradas = [c for c in aliases if c in df.columns]
    if encontradas:
        return encontradas[0]
    if obrigatoria:
        raise ValueError(f"Nenhuma coluna encontrada para {aliases}. Colunas reais: {df.columns}")
    return None

id_inspecao_col = coluna_existente(bronze, ["id_inspecao", "inspecao_id", "id_inspeção"])
id_uc_col = coluna_existente(bronze, ["id_uc", "uc", "unidade_consumidora"])
data_col = coluna_existente(bronze, ["data", "data_inspecao", "data_inspeção", "dt_inspecao"])
resultado_col = coluna_existente(bronze, ["resultado", "resultado_inspecao", "resultado_inspeção"])
kwh_col = coluna_existente(bronze, ["kwh_recuperado", "energia_recuperada_kwh", "kwh"], False)
tipo_col = coluna_existente(bronze, ["tipo_irregularidade", "tipo_irregularidade_encontrada"], False)
inspetor_col = coluna_existente(bronze, ["inspetor", "responsavel"], False)
print({"id_inspecao": id_inspecao_col, "id_uc": id_uc_col, "data": data_col, "resultado": resultado_col, "kwh": kwh_col, "tipo": tipo_col, "inspetor": inspetor_col})

## 3. Cabeçalhos repetidos, deduplicação e integridade da UC

**Decisão:** remover cabeçalho repetido antes da deduplicação, deduplicar por `id_inspecao` e remover UCs nulas ou órfãs. Os registros rejeitados continuam disponíveis na Bronze.

In [ ]:
n0 = bronze.count()
sem_cabecalho = bronze.filter(F.col(id_inspecao_col) != F.lit(id_inspecao_col))
n1 = sem_cabecalho.count()
print(f"Cabeçalhos removidos: {n0 - n1} | antes: {n0} | depois: {n1}")
sem_copias = sem_cabecalho.dropDuplicates([id_inspecao_col])
n2 = sem_copias.count()
print(f"Cópias removidas: {n1 - n2} | antes: {n1} | depois: {n2}")
ids_uc = spark.table(f"{bronze_schema}.unidades_consumidoras").select("id_uc").distinct()
validos = (sem_copias.filter(F.col(id_uc_col).isNotNull()).join(ids_uc, sem_copias[id_uc_col] == ids_uc.id_uc, "left_semi"))
n3 = validos.count()
print(f"Inspeções sem UC ou UC órfã removidas: {n2 - n3} | antes: {n2} | depois: {n3}")

## 4. Datas, decimais e campos manuais

**Decisões:** converter os três formatos de data observados; trocar vírgula decimal por ponto; transformar `-` em nulo em `tipo_irregularidade`; preservar o texto original do resultado antes da padronização; remover espaços do nome do inspetor.

In [ ]:
def data_flexivel(coluna):
    c = F.col(coluna).cast("string")
    return F.coalesce(F.to_date(c, "yyyy-MM-dd"), F.to_date(c, "dd.MM.yyyy"), F.to_date(c, "MM-dd-yyyy"), F.to_date(c, "dd/MM/yyyy"))

ins = (validos.withColumnRenamed(id_inspecao_col, "id_inspecao").withColumnRenamed(id_uc_col, "id_uc")
       .withColumn("data_inspecao", data_flexivel(data_col))
       .withColumn("resultado_original", F.col(resultado_col).cast("string"))
       .withColumn("resultado_padronizado", F.upper(F.trim(F.col(resultado_col).cast("string"))))
       .withColumn("resultado_padronizado", F.when(F.col("resultado_padronizado").rlike("(^CONFIRMADA$|IRREGULARIDADE.*CONFIRMADA)"), "IRREGULARIDADE_CONFIRMADA").when(F.col("resultado_padronizado").rlike("SEM.*IRREGULARIDADE|NÃO.*CONFIRMADA"), "SEM_IRREGULARIDADE").otherwise(F.col("resultado_padronizado")))
       .withColumn("irregularidade_confirmada", F.col("resultado_padronizado") == "IRREGULARIDADE_CONFIRMADA"))
if kwh_col:
    ins = ins.withColumn("kwh_recuperado_original", F.col(kwh_col).cast("string"))
    ins = ins.withColumn("kwh_recuperado", F.regexp_replace(F.trim(F.col(kwh_col).cast("string")), ",", ".").cast("decimal(12,2)"))
if tipo_col:
    ins = ins.withColumn("tipo_irregularidade", F.when(F.trim(F.col(tipo_col)) == "-", F.lit(None)).otherwise(F.upper(F.trim(F.col(tipo_col)))))
if inspetor_col:
    ins = ins.withColumn("inspetor", F.trim(F.col(inspetor_col)))
inspecoes_silver = ins.select(*[c for c in ins.columns if c not in [data_col, resultado_col] or c in ["data_inspecao", "resultado_padronizado"]])
inspecoes_silver.printSchema()

## 5. Checagens de qualidade e gravação

As checagens são registradas em `silver.dq_log` antes da decisão de gravar a Silver.

In [ ]:
dq_results = []
def dq_check(table_name, check_name, df, condition):
    total = df.count(); failed = df.filter(~condition).count(); passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total, failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total}")
def dq_check_unique(table_name, check_name, df, keys):
    total = df.count(); failed = df.groupBy(*keys).count().filter("count > 1").count(); passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total, failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed} chaves duplicadas")

dq_check_unique("silver.inspecoes", "id_inspecao único", inspecoes_silver, ["id_inspecao"])
dq_check("silver.inspecoes", "id_inspecao não nulo", inspecoes_silver, F.col("id_inspecao").isNotNull())
dq_check("silver.inspecoes", "id_uc não nulo", inspecoes_silver, F.col("id_uc").isNotNull())
dq_check("silver.inspecoes", "data_inspecao não nula", inspecoes_silver, F.col("data_inspecao").isNotNull())
dq_check("silver.inspecoes", "aba_origem não nula", inspecoes_silver, F.col("aba_origem").isNotNull())
if kwh_col:
    dq_check("silver.inspecoes", "kwh_recuperado não negativo", inspecoes_silver, F.col("kwh_recuperado").isNull() | (F.col("kwh_recuperado") >= 0))

dq_df = spark.createDataFrame(dq_results)
dq_df.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.dq_log")
falhas = [r for r in dq_results if not r.passed]
if falhas:
    raise Exception(f"Silver não gravada: {len(falhas)} checagem(ns) falharam. Consulte silver.dq_log.")
inspecoes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_silver)
print(f"Gravada: {tabela_silver}")

## 6. Conferência Bronze × Silver

In [ ]:
display(bronze.select(id_inspecao_col, id_uc_col, data_col, resultado_col, "aba_origem").limit(10))
display(spark.table(tabela_silver).limit(10))